# PICKO · Notebook 2 — Tool Families, Finetuning & the Research Story

Explore the scientific tool families, finetune Needle (or reuse a checkpoint), and measure
**two things separately**:
- **Tool-function selection** — did PICKO pick the right *tool*?  (`selection_acc`, `name_f1`)
- **Parameter extraction** — given the right tool, did it extract the right *arguments*?  (`args_exact_acc`, `param_f1`)

Then tell the story across the three research dimensions:
- **D2** parameter extraction · **D3** disambiguation of similar tools · **D1** tool-set size.

> Kernel: **PICKO (.venv)**. Finetuning is **reuse-on-demand** — an existing checkpoint is loaded
> automatically; set `RUN_FINETUNE=True` to retrain (~40 min on CPU).

### 1 · Setup

In [ ]:
import os, sys, json, glob, subprocess
ROOT = os.path.abspath("..")
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid")
except Exception:
    sns = None
from scripts.tool_catalog import Catalog, family_of
cat = Catalog()
print(f"Catalog: {len(cat.tools)} tools · {len(cat.list_families())} families")

## 2 · Explore the tool families

The catalog (`full_tools_53tools_11products.json`) holds **75 tools in 11 families/products**,
grouped into **4 categories** (via `tools_metadata.csv`). Parameter counts vary widely — that
variation is what stresses **D2** (parameter extraction).

In [ ]:
df = cat.as_dataframe()
display(df.groupby(["category","family"]).agg(n_tools=("tool","size"),
        avg_params=("total_params","mean")).round(1))

fig, ax = plt.subplots(1, 2, figsize=(13,4))
fam = cat.list_families()
ax[0].barh(list(fam)[::-1], [fam[f]["count"] for f in fam][::-1], color="#4C72B0")
ax[0].set_title("tools per family")
catcount = df.groupby("category")["tool"].size().sort_values()
ax[1].barh(catcount.index, catcount.values, color="#55A868")
ax[1].set_title("tools per category"); plt.tight_layout(); plt.show()

plt.figure(figsize=(7,3.5))
plt.hist(df["total_params"], bins=range(0, df.total_params.max()+2), color="#C44E52", alpha=.8, label="total")
plt.hist(df["required_params"], bins=range(0, df.total_params.max()+2), color="#4C72B0", alpha=.8, label="required")
plt.xlabel("# parameters"); plt.ylabel("# tools"); plt.legend(); plt.title("parameter-count distribution (D2 difficulty)")
plt.tight_layout(); plt.show()

## 3 · Choose what to analyze

Pick the **data** (a JSONL from Notebook 1) and the **checkpoint**. Defaults point to the completed
8-tool run so the notebook is runnable immediately and reproduces the known numbers. To study the
11-product families, generate data in Notebook 1, point `DATA_JSONL` at it, and set `RUN_FINETUNE=True`.

In [ ]:
DATA_JSONL = os.path.join(ROOT, "data", "picko_subset.jsonl")   # <- from Notebook 1
CKPT_GLOB  = os.path.join(ROOT, "checkpoints", "needle_finetuned_*_best.pkl")
RUN_FINETUNE = False          # flip to True to retrain on DATA_JSONL (~40 min CPU)
EPOCHS, BATCH = 3, 32
EVAL_SUBSAMPLE = None         # e.g. 40 to speed up the D1 sweep on CPU; None = full test set

examples = [json.loads(l) for l in open(DATA_JSONL) if l.strip()]
from needle.training.finetune import _per_tool_split
train, val, test = _per_tool_split(examples)
print(f"{len(examples)} examples -> train {len(train)} / val {len(val)} / test {len(test)}")
tools_in_data = sorted({a['name'] for e in examples for a in json.loads(e['answers']) if a.get('name')})
print("tools:", tools_in_data)

### 4 · Finetune (reuse-on-demand)

In [ ]:
if RUN_FINETUNE:
    print("Finetuning… (this is slow on CPU)")
    subprocess.run(["needle","finetune",DATA_JSONL,"--epochs",str(EPOCHS),"--batch-size",str(BATCH)],
                   cwd=ROOT, check=True)
ckpts = sorted(glob.glob(CKPT_GLOB), key=os.path.getmtime)
assert ckpts, "No finetuned checkpoint found — run Notebook 1 + set RUN_FINETUNE=True"
CKPT = ckpts[-1]
print("Using checkpoint:", os.path.basename(CKPT))

## 5 · Run inference (base vs finetuned)

Greedy-decode both the base (un-finetuned) Needle and our finetuned PICKO on the held-out test set.
Cached so later cells reuse the predictions. **Slow on CPU** (~a few min per model).

In [ ]:
from scripts.picko_eval import load_model, predict, evaluate, confusion, build_tools_override, base_checkpoint
from tqdm.auto import tqdm

test_eval = test if EVAL_SUBSAMPLE is None else test[:EVAL_SUBSAMPLE]

def run(ckpt, label):
    m, p, tok = load_model(ckpt)
    bar = tqdm(total=len(test_eval), desc=label, unit="ex")
    preds = predict(m, p, tok, test_eval, progress=lambda i,n: bar.update(i-bar.n))
    bar.close()
    return (m, p, tok), preds

(_, _, tok), base_preds = run(base_checkpoint(), "base")
ft_bundle, ft_preds = run(CKPT, "finetuned")

## 6 · Core result — tool selection vs parameter extraction  (D2)

The headline of PICKO: the base model usually **names** the right tool but **fumbles arguments**;
finetuning mostly fixes **parameter extraction**.

In [ ]:
base_m = evaluate(test_eval, base_preds, family_of=family_of)
ft_m   = evaluate(test_eval, ft_preds,   family_of=family_of)

summary = pd.DataFrame({
    "metric": ["selection_acc (tool function)","name_f1 (tool function)",
               "args_exact_acc (param extraction)","param_f1 (param extraction)",
               "call_exact (both)","abstain_acc"],
    "base":      [base_m[k] for k in ["selection_acc","name_f1","args_exact_acc","param_f1","call_exact","abstain_acc"]],
    "finetuned": [ft_m[k]   for k in ["selection_acc","name_f1","args_exact_acc","param_f1","call_exact","abstain_acc"]],
})
summary["delta"] = (summary["finetuned"].astype(float) - summary["base"].astype(float)).round(3)
display(summary)

ax = summary.set_index("metric")[["base","finetuned"]].astype(float).plot.bar(
        figsize=(10,4), color=["#B0B0B0","#4C72B0"])
ax.set_title("Base vs Finetuned — selection vs parameter extraction"); ax.set_ylim(0,1)
plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.show()

### 6b · Per-tool and per-family breakdown

In [ ]:
def per_frame(m, key):
    return pd.DataFrame([{ "name": k, "n": v["n"], "selection_acc": v["selection_acc"],
        "args_exact_acc": v["args_exact_acc"], "param_f1": v["param_f1"]} for k,v in m[key].items()])
pt = per_frame(ft_m,"per_tool").merge(cat.as_dataframe()[["tool","total_params"]],
                                      left_on="name", right_on="tool", how="left").sort_values("total_params")
display(pt)

plt.figure(figsize=(7,4))
plt.scatter(pt["total_params"], pt["args_exact_acc"], s=60, color="#4C72B0")
for _,r in pt.iterrows(): plt.annotate(r["name"].split("_")[0], (r["total_params"], r["args_exact_acc"]), fontsize=8)
plt.xlabel("# parameters in tool"); plt.ylabel("param-extraction accuracy (finetuned)")
plt.title("D2: parameter extraction vs tool complexity"); plt.tight_layout(); plt.show()

## 7 · D3 — disambiguation of similar tools

When PICKO picks the **wrong** tool, *which* tool does it pick? Confusions concentrate among
semantically similar tools (same category / same action across sources).

In [ ]:
conf = confusion(test_eval, ft_preds)
labels = sorted(set(conf) | {p for row in conf.values() for p in row})
M = pd.DataFrame(0, index=sorted(conf), columns=labels)
for r, row in conf.items():
    for p, n in row.items(): M.loc[r, p] = n
plt.figure(figsize=(1.1*len(labels)+2, 0.6*len(M)+2))
if sns: sns.heatmap(M, annot=True, fmt="d", cmap="Blues", cbar=False)
else: plt.imshow(M.values, cmap="Blues"); plt.xticks(range(len(labels)),labels,rotation=90); plt.yticks(range(len(M)),M.index)
plt.xlabel("predicted tool"); plt.ylabel("reference tool"); plt.title("D3 confusion (finetuned)")
plt.tight_layout(); plt.show()

## 8 · D1 — tool-set size

**Cheap proxy (no retraining):** keep the finetuned model fixed and vary how many tools are *offered*
at eval time — the correct tool plus `k−1` distractors drawn from the full 75-tool catalog. Measure
tool-selection accuracy as `k` grows.

In [ ]:
ft_model, ft_params, _ = ft_bundle
Ks = [5, 10, 20, 50]
sel_by_k = []
for k in Ks:
    ov = build_tools_override(test_eval, k, cat.tools, seed=0)
    bar = tqdm(total=len(test_eval), desc=f"k={k}", unit="ex")
    preds_k = predict(ft_model, ft_params, tok, test_eval, tools_override=ov,
                      progress=lambda i,n: bar.update(i-bar.n))
    bar.close()
    sel_by_k.append(evaluate(test_eval, preds_k, family_of=family_of)["selection_acc"])
plt.figure(figsize=(7,4))
plt.plot(Ks, sel_by_k, "o-", color="#4C72B0", label="cheap (eval-time distractors)")
plt.xlabel("# tools offered (k)"); plt.ylabel("tool-selection accuracy")
plt.title("D1: does accuracy degrade as the tool set grows?"); plt.ylim(0,1); plt.legend(); plt.tight_layout(); plt.show()
pd.DataFrame({"k":Ks,"selection_acc":sel_by_k})

### 8b · D1 full retrain (optional, heavy)

The rigorous version: finetune a *separate* model for each tool-set size and evaluate. Many slow CPU
runs — gated behind `RUN_D1_FULL`. Overlay on the cheap curve to check the proxy.

In [ ]:
RUN_D1_FULL = False   # set True to retrain per size (very slow)
if RUN_D1_FULL:
    full = {}
    for k in Ks:
        names_k = tools_in_data[:k] if k <= len(tools_in_data) else [t["name"] for t in cat.tools[:k]]
        sub = [e for e in examples if json.loads(e["answers"]) and json.loads(e["answers"])[0]["name"] in set(names_k)]
        path = os.path.join(ROOT, "data", f"picko_d1_{k}.jsonl")
        open(path,"w").write("\n".join(json.dumps(e) for e in sub))
        subprocess.run(["needle","finetune",path,"--epochs",str(EPOCHS),"--batch-size",str(BATCH)], cwd=ROOT, check=True)
        ck = sorted(glob.glob(CKPT_GLOB), key=os.path.getmtime)[-1]
        mk, pk, tk = load_model(ck)
        _,_,tst = _per_tool_split(sub)
        pr = predict(mk, pk, tk, tst)
        full[k] = evaluate(tst, pr, family_of=family_of)["selection_acc"]
    print(full)
else:
    print("RUN_D1_FULL is False — skipping the heavy retrain sweep.")

## 9 · The research story

*(Fill in with your numbers as you run the cells.)*

**PICKO — a 26M tool-picker for scientific agents.** Needle already knows *which* tool a scientific
request needs (high `name_f1` even before finetuning) — the hard part is producing a **valid, correctly
parameterized call**. Finetuning on a few hundred examples per tool closes most of that gap:

- **D2 (parameter extraction):** the largest gains are here — `args_exact_acc` and `param_f1` jump,
  especially for multi-parameter tools (see §6b). This is where a small specialized model earns its keep.
- **D3 (disambiguation):** residual errors cluster among semantically similar tools (§7) — e.g. search
  tools that differ only by source. This is the frontier for a tool-picker and motivates confidence-based
  routing across categorical PICKO instances.
- **D1 (tool-set size):** selection accuracy vs `k` (§8) shows how far one small model scales before an
  orchestrator should shard the tool set.

**Takeaway:** for single-shot scientific tool calling, a tiny fine-tuned model is a strong, cheap
tool-picker — strongest on parameter extraction, with similar-tool disambiguation and large tool sets
as the open challenges.